# 02 — Data Cleaning & Feature Engineering

**Goal of this notebook:** fix every issue flagged in `1_dataLoading.ipynb`, in an order that doesn't break downstream steps, and save a clean file to `data/processed/` that all later notebooks and the Power BI dashboard will use.

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/healthcare_dataset.csv')
df.shape

(56542, 15)

In [2]:
# Preview unique values across key categorical columns before standardizing —
# helps confirm whether casing/whitespace issues actually exist before fixing them
print(df['Medical Condition'].unique())
print(df['Hospital'].unique())
print(df['Gender'].unique())
print(df['Insurance Provider'].unique())
print(df['Admission Type'].unique())
print(df['Medication'].unique())

['Obesity' 'Injury' 'Asthma' 'Cancer' 'Arthritis' 'Diabetes'
 'Hypertension']
['Burns-Dominguez' 'Inc Scott' 'PLC Jenkins' ...
 'Moran Martinez, Ramirez and' 'Friedman Ltd' 'Wilson-Spencer']
['Female' 'Male']
['Cigna' 'UnitedHealthcare' 'Aetna' 'Blue Cross' 'Medicare']
['Elective' 'Emergency' 'Urgent']
['Penicillin' 'Lipitor' 'Paracetamol' 'Aspirin' 'Ibuprofen']


In [3]:
# Making Name of patient in correct/consistent case format
df['Name'] = df['Name'].str.title()
print(df['Name'].head())

0    Matthew Madden
1    Tanner Raymond
2     Michael Davis
3      Marc Francis
4         Tara Koch
Name: Name, dtype: object


In [4]:
# Changing dtype of date-related columns from text to actual datetime objects
# Without this conversion, we cannot subtract one date from another —
# Python would just be comparing strings character by character, not chronologically
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date'] = pd.to_datetime(df['Discharge Date'])
df.dtypes

Name                          object
Age                            int64
Gender                        object
Blood Type                    object
Medical Condition             object
Date of Admission     datetime64[ns]
Doctor                        object
Hospital                      object
Insurance Provider            object
Billing Amount               float64
Room Number                    int64
Admission Type                object
Discharge Date        datetime64[ns]
Medication                    object
Test Results                  object
dtype: object

In [5]:
# Sanity check: a discharge date can never come before the admission date.
# Any row where this happens is a data entry error, not a real patient record,
# so these rows are removed rather than "fixed" — we have no way to know which
# of the two dates was actually correct
print(f'No. of rows with Discharge date before Admission date: {(df["Discharge Date"] < df["Date of Admission"]).sum()}')
df = df[df['Date of Admission'] <= df['Discharge Date']]
print(f'No. of rows remaining after removing invalid date rows: {len(df)}')

No. of rows with Discharge date before Admission date: 704
No. of rows remaining after removing invalid date rows: 55838


In [6]:
# Feature engineering: creating a new column for length of hospital stay
# This doesn't exist in the raw data but is one of the most important KPIs
# in hospital operations analytics
df['Days Stayed'] = (df['Discharge Date'] - df['Date of Admission']).dt.days.astype(int)

In [7]:
df['Days Stayed'].describe()
# Quick check: min should be >= 0, max should be a realistic number of days,
# not something absurd like 5000

count    55838.000000
mean         7.834987
std          4.983217
min          1.000000
25%          4.000000
50%          7.000000
75%          9.000000
max         21.000000
Name: Days Stayed, dtype: float64

In [8]:
# Trimming whitespace and standardizing casing across categorical text columns.
# Without this, groupby('Medical Condition') would treat 'Diabetes' and 'diabetes'
# as two separate groups, silently producing wrong counts and averages
df['Admission Type'] = df['Admission Type'].str.strip().str.title()
df['Medical Condition'] = df['Medical Condition'].str.strip().str.title()
df['Test Results'] = df['Test Results'].str.strip().str.title()
df['Gender'] = df['Gender'].str.strip().str.title()

In [9]:
# Removing exact duplicate rows — a duplicated patient record would otherwise
# be double-counted in every aggregation done later (counts, averages, sums)
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df)} rows')
print('Remaining rows: ', len(df))

Removed 149 rows
Remaining rows:  55689


In [10]:
df.isnull().sum()
# Re-checking for missing values after the cleaning steps above —
# none of the operations so far should have introduced new nulls,
# this just confirms that

Name                    0
Age                     0
Gender                  0
Blood Type              0
Medical Condition       0
Date of Admission       0
Doctor                  0
Hospital                0
Insurance Provider      0
Billing Amount        712
Room Number             0
Admission Type          0
Discharge Date          0
Medication              0
Test Results            0
Days Stayed             0
dtype: int64

In [11]:
# Treating missing values in Billing Amount column (if any exist)
# Using median rather than mean — billing data tends to have outliers
# (a handful of very expensive cases), and the mean would be pulled
# upward by those, giving an unrealistically high fill value for typical patients
missing_count = df['Billing Amount'].isnull().sum()
print('No. of missing value found', missing_count)
print(f'Total {missing_count*100/len(df)}% of billing data have missing value')
df['Billing Amount'] = df['Billing Amount'].fillna(df['Billing Amount'].median())

No. of missing value found 712
Total 1.2785289734058791% of billing data have missing value


In [12]:
# Treating negative values in Billing Amount.
# We deliberately do NOT use .abs() here — a negative value could be a sign-flip
# data entry error, OR it could represent a refund/credit record that shouldn't
# be in a patient charge analysis at all. We have no way to tell which from the
# data alone, so flipping the sign would mean fabricating charge amounts that
# may never have actually occurred. Removing these rows is the safer, defensible choice.
negative_count = (df['Billing Amount'] < 0).sum()
print('No. of negative value found', negative_count)
print(f'Total {negative_count*100/len(df)}% of billing data have -ve value')

df = df[df['Billing Amount'] >= 0]
print('No of rows remaining', len(df))

No. of negative value found 336
Total 0.6033507514949092% of billing data have -ve value
No of rows remaining 55353


In [13]:
# Handling negative age values — age can never be negative, so any such row
# is a data entry error and is removed
negative_age_count = (df['Age'] < 0).sum()
print('No. of negative age values found:', negative_age_count)
df = df[df['Age'] >= 0]
print('No. of rows remaining:', len(df))

No. of negative age values found: 341
No. of rows remaining: 55012


In [14]:
# Making Age column valid — ages above 110 are not realistic for this dataset
# and are treated as data entry errors rather than genuine extreme cases
before = len(df)
df = df[df['Age'] <= 110]
print(f'No. of patients left: {len(df)}')
print(f'Rows removed: {before - len(df)}')

No. of patients left: 54646
Rows removed: 366


In [15]:
# Handling outliers in Billing Amount using the IQR method.
#
# IMPORTANT DESIGN CHOICE: outliers are calculated PER medical condition,
# not globally across the whole dataset. Cancer billing naturally runs into
# tens of thousands while Asthma billing runs into low thousands — a single
# global threshold would flag almost every Cancer case as an "outlier" simply
# because Cancer is expensive, not because those rows are actually wrong.
# Grouping by condition first means each diagnosis is judged against its own
# realistic price range.
#
# Implementation note: rather than groupby().apply() with a function that
# returns a filtered DataFrame (whose handling of the grouping column has
# changed across pandas versions — 2.x kept it, 3.0 dropped it, and the
# include_groups flag that bridged the two was removed again in later 3.x
# releases), we compute the lower/upper bound for every row directly via
# transform(). This sidesteps the issue entirely: each row gets its own
# condition's bounds attached as new columns, then a single boolean mask
# filters everything at once. Same result, no version-dependent behavior.

Q1 = df.groupby('Medical Condition')['Billing Amount'].transform('quantile', 0.25)
Q3 = df.groupby('Medical Condition')['Billing Amount'].transform('quantile', 0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

is_outlier = (df['Billing Amount'] < lower_bound) | (df['Billing Amount'] > upper_bound)
print('Outliers found per condition:')
print(df.loc[is_outlier, 'Medical Condition'].value_counts())

df_no_outliers = df[~is_outlier].reset_index(drop=True)

Outliers found per condition:
Medical Condition
Cancer          192
Asthma           75
Obesity          68
Arthritis        50
Injury           47
Diabetes         39
Hypertension     24
Name: count, dtype: int64


In [16]:
# Final check of the cleaned dataframe before saving
print(df_no_outliers.shape)
print(df_no_outliers.isnull().sum())
print(df_no_outliers.describe())

(54151, 16)
Name                  0
Age                   0
Gender                0
Blood Type            0
Medical Condition     0
Date of Admission     0
Doctor                0
Hospital              0
Insurance Provider    0
Billing Amount        0
Room Number           0
Admission Type        0
Discharge Date        0
Medication            0
Test Results          0
Days Stayed           0
dtype: int64
                Age              Date of Admission  Billing Amount  \
count  54151.000000                          54151    54151.000000   
mean      45.832542  2021-10-31 01:27:32.512418816    20784.606284   
min       12.000000            2019-05-06 00:00:00      658.850000   
25%       29.000000            2020-07-27 00:00:00     5392.090000   
50%       46.000000            2021-10-30 00:00:00     8325.730000   
75%       61.000000            2023-02-01 00:00:00    12417.015000   
max       90.000000            2024-05-07 00:00:00   106199.210000   
std       19.319350            

In [17]:
print(df_no_outliers.head())

             Name  Age  Gender Blood Type Medical Condition Date of Admission  \
0  Matthew Madden   58  Female         B+           Obesity        2024-02-05   
1  Tanner Raymond   26  Female         B-            Injury        2022-10-05   
2   Michael Davis   27  Female         B-           Obesity        2020-12-07   
3    Marc Francis   21    Male         B+            Asthma        2023-05-01   
4       Tara Koch   21    Male         A+            Injury        2024-02-05   

           Doctor                    Hospital Insurance Provider  \
0      Heather Ho             Burns-Dominguez              Cigna   
1  Michael Obrien                   Inc Scott   UnitedHealthcare   
2    Willie Smith                 PLC Jenkins              Aetna   
3  William Nelson  Cisneros Short, and Taylor              Cigna   
4      Nancy Ruiz           Erickson-Martinez              Cigna   

   Billing Amount  Room Number Admission Type Discharge Date   Medication  \
0         9925.15          

In [18]:
# Final confirmation that all categorical columns are clean and consistent
print(df_no_outliers['Medical Condition'].unique())
print(df_no_outliers['Hospital'].unique())
print(df_no_outliers['Gender'].unique())
print(df_no_outliers['Insurance Provider'].unique())
print(df_no_outliers['Admission Type'].unique())
print(df_no_outliers['Medication'].unique())

['Obesity' 'Injury' 'Asthma' 'Cancer' 'Arthritis' 'Diabetes'
 'Hypertension']
['Burns-Dominguez' 'Inc Scott' 'PLC Jenkins' ...
 'Moran Martinez, Ramirez and' 'Friedman Ltd' 'Wilson-Spencer']
['Female' 'Male']
['Cigna' 'UnitedHealthcare' 'Aetna' 'Blue Cross' 'Medicare']
['Elective' 'Emergency' 'Urgent']
['Penicillin' 'Lipitor' 'Paracetamol' 'Aspirin' 'Ibuprofen']


In [19]:
# Saving the cleaned dataset — this is the single file every later notebook
# AND the Power BI dashboard will read from. Raw data in data/raw/ is never
# overwritten, so the original source is always preserved.
df_no_outliers.to_csv('../data/processed/healthcare_data_clean.csv', index=False)
print('Saved Successfully')

Saved Successfully


## Cleaning summary

| Step | Rows affected |
|---|---|
| Invalid date rows (discharge before admission) | 704 removed |
| Duplicate rows | 149 removed |
| Negative Billing Amount | 336 removed |
| Negative / invalid Age | 707 removed |
| Billing outliers (per condition, IQR) | 495 removed |

Final dataset: **54,151 rows × 16 columns** (added `Days Stayed`)

Removed: **4.22%** of Data.